In [3]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier


# -----------------------------
# Load data
# -----------------------------
loan_df = pd.read_csv("Task 3 and 4_Loan_Data.csv")

print(loan_df.head())
print(loan_df.info())


# -----------------------------
# FICO bucketing function
# -----------------------------
def create_fico_rating_map(data, num_buckets=5):
    """
    Create FICO score buckets using a decision tree.

    The decision tree finds score boundaries that help separate default
    and non-default borrowers.

    Lower rating means better credit score.
    Example:
    Rating 1 = strongest/highest FICO bucket
    Rating 5 = weakest/lowest FICO bucket
    """

    X = data[["fico_score"]]
    y = data["default"]

    tree = DecisionTreeClassifier(
        max_leaf_nodes=num_buckets,
        min_samples_leaf=20,
        random_state=42
    )

    tree.fit(X, y)

    # Extract split thresholds from tree
    thresholds = tree.tree_.threshold
    boundaries = sorted([t for t in thresholds if t != -2])

    min_score = data["fico_score"].min()
    max_score = data["fico_score"].max()

    # Bucket boundaries
    bucket_edges = [min_score - 1] + boundaries + [max_score + 1]

    # Remove duplicates and sort
    bucket_edges = sorted(list(set(bucket_edges)))

    # Create bucket labels
    data_copy = data.copy()

    data_copy["fico_bucket"] = pd.cut(
        data_copy["fico_score"],
        bins=bucket_edges,
        include_lowest=True
    )

    # Rank buckets by average FICO score
    bucket_summary = (
        data_copy
        .groupby("fico_bucket", observed=True)
        .agg(
            min_fico=("fico_score", "min"),
            max_fico=("fico_score", "max"),
            avg_fico=("fico_score", "mean"),
            borrowers=("fico_score", "count"),
            defaults=("default", "sum"),
            default_rate=("default", "mean")
        )
        .reset_index()
    )

    # Higher FICO = better rating
    bucket_summary = bucket_summary.sort_values("avg_fico", ascending=False).reset_index(drop=True)

    bucket_summary["rating"] = range(1, len(bucket_summary) + 1)

    return bucket_edges, bucket_summary


# -----------------------------
# Function to assign rating to new borrower
# -----------------------------
def fico_to_rating(fico_score, bucket_summary):
    """
    Map a FICO score to a rating.
    Lower rating means better credit quality.
    """

    for _, row in bucket_summary.iterrows():
        if row["min_fico"] <= fico_score <= row["max_fico"]:
            return int(row["rating"])

    # If score is above highest known range, assign best rating
    if fico_score > bucket_summary["max_fico"].max():
        return 1

    # If score is below lowest known range, assign worst rating
    if fico_score < bucket_summary["min_fico"].min():
        return int(bucket_summary["rating"].max())

    return None


# -----------------------------
# Create rating map
# -----------------------------
num_buckets = 5

bucket_edges, rating_map = create_fico_rating_map(
    loan_df,
    num_buckets=num_buckets
)

print("Bucket edges:")
print(bucket_edges)

print("\nRating map:")
print(rating_map)


# -----------------------------
# Sample test
# -----------------------------
sample_fico = 650

sample_rating = fico_to_rating(sample_fico, rating_map)

print(f"\nFICO score: {sample_fico}")
print(f"Assigned rating: {sample_rating}")

   customer_id  credit_lines_outstanding  loan_amt_outstanding  \
0      8153374                         0           5221.545193   
1      7442532                         5           1958.928726   
2      2256073                         0           3363.009259   
3      4885975                         0           4766.648001   
4      4700614                         1           1345.827718   

   total_debt_outstanding       income  years_employed  fico_score  default  
0             3915.471226  78039.38546               5         605        0  
1             8228.752520  26648.43525               2         572        1  
2             2027.830850  65866.71246               4         602        0  
3             2501.730397  74356.88347               5         612        0  
4             1768.826187  23448.32631               6         631        0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column                 